# DEH 30-Day PySpark Challenge
## Day 29 — Practice Set 8: Company-Style Problems

**Instructions:**
1. `File → Save a copy in Drive` first
2. Attempt each problem before checking the reference
3. Reference solutions follow each problem

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder.appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id", StringType(), True), StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True), StructField("order_date", DateType(), True),
    StructField("quantity", IntegerType(), True), StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", IntegerType(), True), StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True), StructField("region", StringType(), True)
])
customers_schema = StructType([
    StructField("customer_id", StringType(), True), StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True), StructField("email", StringType(), True),
    StructField("city", StringType(), True), StructField("state", StringType(), True),
    StructField("country", StringType(), True), StructField("signup_date", DateType(), True),
    StructField("segment", StringType(), True)
])
products_schema = StructType([
    StructField("product_id", StringType(), True), StructField("product_name", StringType(), True),
    StructField("category", StringType(), True), StructField("sub_category", StringType(), True),
    StructField("unit_price", DoubleType(), True), StructField("cost_price", DoubleType(), True),
    StructField("supplier", StringType(), True), StructField("stock_quantity", IntegerType(), True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
products_df  = spark.read.option("header","true").schema(products_schema).csv(f"s3a://{BUCKET}/data/products.csv")

print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()} | Products: {products_df.count()}")

---
## Problem 1 (Amazon-style) — Customers who ordered the same product 3+ times

Return customer_id, product_id, times_ordered for repeat purchases >= 3.

In [ ]:
# Your attempt


### Reference Solution — Problem 1

In [ ]:
orders_df.groupBy("customer_id", "product_id").agg(
    F.count("order_id").alias("times_ordered")
).filter(F.col("times_ordered") >= 3) \
 .orderBy(F.col("times_ordered").desc()) \
 .show()

---
## Problem 2 (Netflix-style) — "Binge" detection: 2+ orders same day

Return customer_id, order_date, orders_that_day for same-day multi-order customers.

In [ ]:
# Your attempt


### Reference Solution — Problem 2

In [ ]:
orders_df.groupBy("customer_id", "order_date").agg(
    F.count("order_id").alias("orders_that_day")
).filter(F.col("orders_that_day") >= 2) \
 .orderBy(F.col("orders_that_day").desc()) \
 .show()

---
## Problem 3 (Uber-style) — Supply-demand mismatch by region

Demand (qty ordered) vs Supply (total stock). Flag region 'high_demand' if demand > 50% of supply.

In [ ]:
# Your attempt


### Reference Solution — Problem 3

Note: total stock is a single company-wide number (not region-specific in our dataset) — so this compares each region's demand against the shared total supply pool, a reasonable simplification.

In [ ]:
total_stock = products_df.agg(F.sum("stock_quantity")).collect()[0][0]

regional_demand = orders_df.groupBy("region").agg(
    F.sum("quantity").alias("total_demand")
)

regional_demand.withColumn("total_supply", F.lit(total_stock)) \
    .withColumn(
        "demand_status",
        F.when(F.col("total_demand") > 0.5 * F.col("total_supply"), "high_demand").otherwise("normal")
    ).orderBy(F.col("total_demand").desc()) \
    .show()

---
## Problem 4 (Meta-style) — Funnel: signup to first purchase

Bucket customers: Same Day, Within Week, Within Month, Slow, Never Purchased.

In [ ]:
# Your attempt


### Reference Solution — Problem 4

In [ ]:
first_order = orders_df.groupBy("customer_id").agg(
    F.min("order_date").alias("first_order_date")
)

funnel = customers_df.join(first_order, on="customer_id", how="left") \
    .withColumn("days_to_purchase", F.datediff(F.col("first_order_date"), F.col("signup_date"))) \
    .withColumn(
        "funnel_bucket",
        F.when(F.col("first_order_date").isNull(), "Never Purchased")
         .when(F.col("days_to_purchase") == 0, "Same Day")
         .when(F.col("days_to_purchase") <= 7, "Within Week")
         .when(F.col("days_to_purchase") <= 30, "Within Month")
         .otherwise("Slow")
    )

funnel.groupBy("funnel_bucket").agg(F.count("customer_id").alias("customer_count")) \
    .orderBy(F.col("customer_count").desc()) \
    .show()

---
## Problem 5 (Spotify-style) — Discovery metric: first category purchase by month

First month each customer bought from each category. Count discoveries per calendar month.

In [ ]:
# Your attempt


### Reference Solution — Problem 5

**Plan:**
1. Join orders with products to get category
2. For each (customer, category) pair, find the earliest order_date — this is their 'discovery'
3. Extract the month from that discovery date
4. Group by month, count discoveries

In [ ]:
orders_with_category = orders_df.join(
    products_df.select("product_id", "category"), on="product_id", how="inner"
)

discoveries = orders_with_category.groupBy("customer_id", "category").agg(
    F.min("order_date").alias("discovery_date")
).withColumn("discovery_month", F.date_format("discovery_date", "yyyy-MM"))

discoveries.groupBy("discovery_month").agg(
    F.count("*").alias("num_discoveries")
).orderBy("discovery_month").show(20)

---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*